Connecting to ADLS

In [0]:
display(
    dbutils.fs.ls(
        "abfss://retailsales@harshadls01.dfs.core.windows.net/"
    )
)

path,name,size,modificationTime
abfss://retailsales@harshadls01.dfs.core.windows.net/bronze/,bronze/,0,1790574521000
abfss://retailsales@harshadls01.dfs.core.windows.net/gold/,gold/,0,1790574534000
abfss://retailsales@harshadls01.dfs.core.windows.net/silver/,silver/,0,1790574528000


In [0]:
customers_df = (
    spark.read
         .option("header","true")
         .csv(
             "abfss://retailsales@harshadls01.dfs.core.windows.net/bronze/customers.csv"
         )
)

display(customers_df)

customer_id,customer_name,city
1,Harsh,Noida
2,Rahul,Delhi
3,Amit,Gurgaon
4,Priya,Mumbai
5,Neha,Pune


In [0]:
products_df = (
    spark.read
         .option("header","true")
         .csv(
             "abfss://retailsales@harshadls01.dfs.core.windows.net/bronze/products.csv"
         )
)

display(products_df)

product_id,product_name,category,price
101,Laptop,Electronics,50000
102,Mobile,Electronics,30000
103,Shoes,Fashion,2000
104,Watch,Fashion,5000


In [0]:
orders_df = (
    spark.read
         .option("header","true")
         .csv(
             "abfss://retailsales@harshadls01.dfs.core.windows.net/bronze/orders.csv"
         )
)

display(orders_df)

order_id,customer_id,product_id,quantity
1001,1,101,1
1002,2,102,2
1003,3,103,3
1004,1,104,2
1005,5,102,1


## Data Cleaning

In [0]:
customers_silver_df = (
    customers_df
    .dropDuplicates()
    .fillna("Unknown")
)

display(customers_silver_df)

customer_id,customer_name,city
4,Priya,Mumbai
2,Rahul,Delhi
3,Amit,Gurgaon
1,Harsh,Noida
5,Neha,Pune


In [0]:
from pyspark.sql.functions import col

In [0]:
products_silver_df = (
    products_df
    .dropDuplicates()
    .withColumn(
        "price",
        col("price").cast("integer")
    )
)

display(products_silver_df)

product_id,product_name,category,price
102,Mobile,Electronics,30000
101,Laptop,Electronics,50000
103,Shoes,Fashion,2000
104,Watch,Fashion,5000


In [0]:
orders_silver_df = (
    orders_df
    .dropDuplicates()
    .withColumn(
        "quantity",
        col("quantity").cast("integer")
    )
)

display(orders_silver_df)

order_id,customer_id,product_id,quantity
1001,1,101,1
1004,1,104,2
1002,2,102,2
1003,3,103,3
1005,5,102,1


## Verify Schema

In [0]:
customers_silver_df.printSchema()
products_silver_df.printSchema()
orders_silver_df.printSchema()

root
 |-- customer_id: string (nullable = false)
 |-- customer_name: string (nullable = false)
 |-- city: string (nullable = false)

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: integer (nullable = true)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)



## Write Silver Delta Tables

In [0]:
customers_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(
        "abfss://retailsales@harshadls01.dfs.core.windows.net/silver/customers_delta"
    )

In [0]:
products_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(
        "abfss://retailsales@harshadls01.dfs.core.windows.net/silver/products_delta"
    )

In [0]:
orders_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(
        "abfss://retailsales@harshadls01.dfs.core.windows.net/silver/orders_delta"
    )